# ACM 模式学习

说明：
区别于力扣的核心代码模式，ACM模式是一种更加原始的评测模式。简单讲就是一种需要自行处理输入输出的模式。

## 1、系统设置检查

说明：
检查虚拟环境以及相应的库

In [1]:
import sys
from collections import Counter, defaultdict, deque
import heapq
import bisect

print(sys.executable)
print(sys.version)

f:\LC-Hot100\.venv\Scripts\python.exe
3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]


## 2、ACM 输入的命令讲解

说明：看起来复杂，本质上只有三件事情
- 读取文本
- 按题目规则分割
- 转化成需要的数据

### 2.1 读取文本

说明：一般有两种方式
- `input()`：Python 内置函数
- `sys.stdin`：要先调用库，稍微复杂一点，但功能更多


##### Step 0：先构建一个辅助函数在 notebook 中模拟 py 脚本行为

In [2]:
import builtins
import sys
from pathlib import Path
from contextlib import contextmanager
from unittest.mock import patch


@contextmanager
def use_input_file(path):
    """临时从 UTF-8 文件读取输入，模拟常见的脚本输入行为。"""
    with open(path, "r", encoding="utf-8") as stream:

        def file_input(prompt=""):
            # 模拟 input() 的提示文字
            if prompt:
                print(prompt, end="", flush=True)

            line = stream.readline()

            # 模拟原生 input() 的 EOF 行为
            if line == "":
                raise EOFError("EOF when reading a line")

            # 只去掉行末换行，保留其他空白
            if line.endswith("\n"):
                line = line[:-1]

            return line

        # 临时替换标准输入和 Notebook 使用的内置 input
        with patch.object(sys, "stdin", stream):
            with patch.object(builtins, "input", file_input):
                yield


SAMPLE = Path("input_demo.txt") # 注意文件路径，确保文件存在

##### Step 1：使用 `input()` 读取一行文本

- `input()` 是 Python 内置函数，无需导入。
- 每次读取一行，返回 `str` 字符串。
- 自动去掉行末换行符，保留其他空格。
- 空行返回空字符串 `''`。
- 输入结束后再次读取，会抛出 `EOFError`。

下面通过辅助函数从文件读取，用 `repr()` 展示换行和空字符串。

In [3]:
with use_input_file(SAMPLE):
    for _ in range(4):
        line = input()  # 读完四行
        print(repr(line), type(line).__name__)

    try:
        input()  # 再读取一次
    except EOFError:
        print("输入已结束，触发 EOFError")

'10 20' str
'' str
'hello' str
'中文' str
输入已结束，触发 EOFError


##### Step 2：使用 `sys.stdin.readline()` 读取一行文本

- `sys` 是 Python 标准库模块，需要 `import sys`，无需额外安装。
- `sys.stdin` 是标准输入流，`readline()` 是它的读取方法。
- 每次读取一行，返回 `str` 字符串。
- 与原生 `input()` 不同，它会保留行末换行符。
- 空行通常返回 `'\n'`；到达输入末尾 EOF 时返回 `''`，不会抛出 `EOFError`。

In [4]:
import sys

with use_input_file(SAMPLE):
    for _ in range(5):
        line = sys.stdin.readline()
        print(repr(line), type(line).__name__)

'10 20\n' str
'\n' str
'hello\n' str
'中文' str
'' str


##### Step 3：使用 `sys.stdin.read()` 读取全部剩余文本

- 不带参数时，从当前位置一直读取到输入结束 EOF。
- 返回一个 `str` 字符串，保留空格和换行符。
- 已经读完后再次调用，返回空字符串 `''`。
- 与 `readline()` 每次读取一行不同，`read()` 可以一次读取多行。
- 全部内容会放入内存，输入量特别大时需要考虑内存占用。

In [5]:
with use_input_file(SAMPLE):
    first_line = sys.stdin.readline()
    remaining = sys.stdin.read()

    print("第一行：", repr(first_line))
    print("剩余内容：", repr(remaining))

第一行： '10 20\n'
剩余内容： '\nhello\n中文'


##### Step 4：使用 `sys.stdin.readlines()` 读取所有剩余行

- 从当前位置读取到 EOF，返回 `list[str]`，即字符串列表。
- 列表中的每个元素对应一行，保留该行的换行符。
- 空行也会成为一个元素，通常是 `'\n'`。
- 已经读完后再次调用，返回空列表 `[]`。
- 所有剩余行都会放入内存。

In [6]:
import sys

with use_input_file(SAMPLE):
    lines = sys.stdin.readlines()

    print("读取结果：", repr(lines))
    print("返回类型：", type(lines).__name__)
    print("再次读取：", sys.stdin.readlines())

读取结果： ['10 20\n', '\n', 'hello\n', '中文']
返回类型： list
再次读取： []


##### Step 5：使用 `for` 逐行读取标准输入

- 每次循环得到一行字符串，保留换行符。
- 空行也会被读取。
- 到达 EOF 后，循环自动结束。
- 不需要预先知道行数，也不需要一次把所有行放进列表。

In [7]:
with use_input_file(SAMPLE):
    for line in sys.stdin:
        print(repr(line))

    print("读取结束")

'10 20\n'
'\n'
'hello\n'
'中文'
读取结束


注意：这里是直接遍历输入流；`for line in sys.stdin.readlines()` 则会先读取全部剩余行并建立列表。

In [8]:
with use_input_file(SAMPLE):
    for line in sys.stdin.readlines():
        print(repr(line))

    print("读取结束")

'10 20\n'
'\n'
'hello\n'
'中文'
读取结束


##### Step 6：使用 `while True` 持续读取

- `while True` 是循环语句，本身不读取数据。
- 每轮调用一次读取方法，再判断是否结束。
- 使用 `readline()` 时，通过返回值 `''` 判断 EOF。
- 使用原生 `input()` 时，通过捕获 `EOFError` 判断 EOF。
- 空行与 EOF 不同，不应把空行直接当作输入结束。

In [9]:
with use_input_file(SAMPLE):
    while True:
        line = sys.stdin.readline()

        if line == "":
            break

        print(repr(line))

    print("读取结束")

'10 20\n'
'\n'
'hello\n'
'中文'
读取结束


In [10]:
with use_input_file(SAMPLE):
    while True:
        try:
            line = input()
        except EOFError:
            break

        print(repr(line))

    print("读取结束")

'10 20'
''
'hello'
'中文'
读取结束


注意：如果没有 `except EOFError`，程序会报错

In [11]:
# with use_input_file(SAMPLE):
#     while True:
#         # try:
#         line = input()
#         # except EOFError:
#         #     break

#         print(repr(line))

#     print("读取结束")

##### Step 7：使用 `sys.stdin.buffer.readline()` 读取一行字节

- `sys.stdin.buffer` 是标准输入的底层二进制缓冲流。
- 每次读取一行，返回 `bytes`，不是 `str`。
- 保留行末换行字节，不做文本模式的换行转换。
- 到达 EOF 时返回 `b''`。
- `.buffer` 是属性，后面的 `readline()` 才负责读取。

In [12]:
with use_input_file(SAMPLE):
    while True:
        line = sys.stdin.buffer.readline()

        if line == b"":
            break

        print("字节：", repr(line))
        print("类型：", type(line).__name__)
        print("解码：", repr(line.decode("utf-8")))

字节： b'10 20\r\n'
类型： bytes
解码： '10 20\r\n'
字节： b'\r\n'
类型： bytes
解码： '\r\n'
字节： b'hello\r\n'
类型： bytes
解码： 'hello\r\n'
字节： b'\xe4\xb8\xad\xe6\x96\x87'
类型： bytes
解码： '中文'


说明：

中文会以编码后的字节形式显示；用匹配的 UTF-8 编码解码后，得到中文字符串。

如果文件使用 Windows 换行，字节中可能出现 `b'\r\n'`；使用 LF 换行则是 `b'\n'`。

##### Step 8：使用 `sys.stdin.buffer.read()` 读取全部剩余字节

- 不带参数时，从当前位置读取到 EOF。
- 返回一个 `bytes` 对象，保留原始字节。
- 已经读完后再次调用，返回 `b''`。
- `.buffer` 决定读取字节，`read()` 决定读取全部剩余内容。
- 全部剩余内容会放入内存。

In [13]:
with use_input_file(SAMPLE):
    data = sys.stdin.buffer.read()

    print("字节：", repr(data))
    print("类型：", type(data).__name__)
    print("解码：", repr(data.decode("utf-8")))
    print("再次读取：", repr(sys.stdin.buffer.read()))

字节： b'10 20\r\n\r\nhello\r\n\xe4\xb8\xad\xe6\x96\x87'
类型： bytes
解码： '10 20\r\n\r\nhello\r\n中文'
再次读取： b''


##### Step 9：使用 `sys.stdin.buffer.readlines()` 读取所有剩余字节行

- 从当前位置读取到 EOF，返回 `list[bytes]`，即字节串列表。
- 每个元素对应一行，保留行末换行字节。
- 空行也会成为一个元素，例如 `b'\n'`。
- 已经读完后再次调用，返回空列表 `[]`。
- 所有剩余行都会放入内存。

In [14]:
with use_input_file(SAMPLE):
    lines = sys.stdin.buffer.readlines()

    print("读取结果：", repr(lines))
    print("返回类型：", type(lines).__name__)

    for line in lines:
        print(
            "元素类型：", type(line).__name__,
            "解码结果：", repr(line.decode("utf-8")),
            
        )

    print("再次读取：", sys.stdin.buffer.readlines())

读取结果： [b'10 20\r\n', b'\r\n', b'hello\r\n', b'\xe4\xb8\xad\xe6\x96\x87']
返回类型： list
元素类型： bytes 解码结果： '10 20\r\n'
元素类型： bytes 解码结果： '\r\n'
元素类型： bytes 解码结果： 'hello\r\n'
元素类型： bytes 解码结果： '中文'
再次读取： []


##### Step 10：使用 `for` 逐行读取字节

- `for line in sys.stdin.buffer` 每次读取一行字节。
- 每次得到的 `line` 类型为 `bytes`，保留行末换行字节。
- 空行也会被读取，例如 `b'\n'`。
- 到达 EOF 后自动结束循环。
- 不需要先用 `readlines()` 将所有行放进列表。

In [15]:
with use_input_file(SAMPLE):
    for line in sys.stdin.buffer:
        print("字节：", repr(line))
        print("类型：", type(line).__name__)
        print("解码：", repr(line.decode("utf-8")))

    print("读取结束")

字节： b'10 20\r\n'
类型： bytes
解码： '10 20\r\n'
字节： b'\r\n'
类型： bytes
解码： '\r\n'
字节： b'hello\r\n'
类型： bytes
解码： 'hello\r\n'
字节： b'\xe4\xb8\xad\xe6\x96\x87'
类型： bytes
解码： '中文'
读取结束


注意：遍历流和遍历已读出的字节串不同

In [16]:
with use_input_file(SAMPLE):
    for value in sys.stdin.buffer.read():
        print(value)  # 每次得到一个字节的整数值，不是一行

49
48
32
50
48
13
10
13
10
104
101
108
108
111
13
10
228
184
173
230
150
135


##### Step 11：理解 `input = sys.stdin.readline`

- 这是赋值操作，把名字 `input` 绑定到 `readline` 方法。
- 右侧没有括号，因此此时不会读取数据。
- 后续执行 `input()`，实际调用的是 `readline()`。
- 因此会保留换行符，EOF 时返回 `''`，不再抛出 `EOFError`。
- 教学中也可以使用 `read_line` 这个名字，避免遮蔽内置 `input`。

In [17]:
# 为了不影响 Notebook 后续单元格，把这个绑定放在函数内部
def demo_input_alias():
    input = sys.stdin.readline  # 只在这个函数内部生效

    for _ in range(5):
        print(repr(input()))


with use_input_file(SAMPLE):
    demo_input_alias()

'10 20\n'
'\n'
'hello\n'
'中文'
''


总结：
| 写法 | 读取范围 | 返回值／每轮得到的类型 | 到达 EOF 时 | 适合的场景 |
|---|---|---|---|---|
| `input()` | 一行 | `str` | 抛出 `EOFError` | 入门、少量输入、已知行数 |
| `sys.stdin.readline()` | 一行 | `str` | 返回 `''` | 逐行读取，手动控制循环 |
| `sys.stdin.read()` | 全部剩余内容 | `str` | 返回 `''` | 一次读入后统一处理 |
| `sys.stdin.readlines()` | 全部剩余行 | `list[str]` | 返回 `[]` | 需要保留各行组成的列表 |
| `for line in sys.stdin` | 逐行读取 | 每轮得到 `str` | 自动结束循环 | 不确定行数，逐行处理 |
| `sys.stdin.buffer.readline()` | 一行字节 | `bytes` | 返回 `b''` | 逐行处理字节数据 |
| `sys.stdin.buffer.read()` | 全部剩余字节 | `bytes` | 返回 `b''` | 批量数字输入，统一处理 |
| `sys.stdin.buffer.readlines()` | 全部剩余字节行 | `list[bytes]` | 返回 `[]` | 需要保留各字节行组成的列表 |
| `for line in sys.stdin.buffer` | 逐行读取字节 | 每轮得到 `bytes` | 自动结束循环 | 不确定行数，逐行处理字节 |

### 2.2 按题目规则分割

说明：这一步只做分割，暂时不转换成整数或者其他需要的格式。

##### 2.2.1：使用 `.split()` 按空白分割，最常用

- 不传参数时，按空白字符分割，包括空格、制表符和换行。
- 连续的空白会作为一个分隔区域处理。
- 忽略字符串两端的空白，不产生空字符串元素。
- 返回字符串列表 `list[str]`。
- 只有空白或没有内容时，返回空列表 `[]`。

In [18]:
examples = [
    "10 20",
    "10   20",
    "  10 20  ",
    "10\t20\n",
    "",
    "   \n",
]

for text in examples:
    print(repr(text), "→", text.split())

'10 20' → ['10', '20']
'10   20' → ['10', '20']
'  10 20  ' → ['10', '20']
'10\t20\n' → ['10', '20']
'' → []
'   \n' → []


In [19]:
with use_input_file(SAMPLE):
    line = input()
    parts = line.split()

    print("原始文本：", repr(line))
    print("分割结果：", parts)
    print("元素类型：", type(parts[0]).__name__)

原始文本： '10 20'
分割结果： ['10', '20']
元素类型： str


注意：`'10'` 和 `'20'` 仍然是字符串

##### 2.2.2：使用 `.split(sep)` 按指定分隔符分割

- `sep` 指定分隔符，例如逗号、冒号或多个字符。
- 分隔符本身不会出现在结果中。
- 连续分隔符、开头或结尾的分隔符，会产生空字符串元素。
- 不会自动删除字段两端的空格。

In [20]:
print("10,20,30".split(","))
# ['10', '20', '30']

print("name:alice".split(":"))
# ['name', 'alice']

print("10::20::30".split("::"))
# ['10', '20', '30']

print("10,,20,".split(","))
# ['10', '', '20', '']

print("10, 20".split(","))
# ['10', ' 20']

['10', '20', '30']
['name', 'alice']
['10', '20', '30']
['10', '', '20', '']
['10', ' 20']


特别：比较 `.split()` 和 `.split(" ")`

In [21]:
text = "  10  20  "

print(text.split())
# ['10', '20']

print(text.split(" "))
# ['', '', '10', '', '20', '', '']

['10', '20']
['', '', '10', '', '20', '', '']


##### 2.2.3：使用 `maxsplit` 限制分割次数

- `maxsplit` 表示最多进行多少次分割。
- 剩余内容作为最后一个元素保留。
- 最多得到 `maxsplit + 1` 个元素。
- 适合“前几个字段固定，后面保留完整文本”的输入。

In [22]:
text = "alice 18 Beijing Chaoyang"

print(text.split())
# ['alice', '18', 'Beijing', 'Chaoyang']

print(text.split(maxsplit=2))
# ['alice', '18', 'Beijing Chaoyang']

['alice', '18', 'Beijing', 'Chaoyang']
['alice', '18', 'Beijing Chaoyang']


In [23]:
text = "name=alice=admin"

print(text.split("=", maxsplit=1))
# ['name', 'alice=admin']

['name', 'alice=admin']


##### 2.2.4：对字节串使用 `.split()`

- `bytes` 同样支持 `.split()`。
- 不传参数时，按 ASCII 空白字节分割，例如空格、制表符和换行。
- 返回字节串列表 `list[bytes]`。
- 指定分隔符时，也必须使用字节串，例如 `b','`。

In [24]:
data = b"10  20\t30\n"

print(data.split())
# [b'10', b'20', b'30']

print(b"10,20,30".split(b","))
# [b'10', b'20', b'30']

[b'10', b'20', b'30']
[b'10', b'20', b'30']


In [25]:
with use_input_file(SAMPLE):
    line = sys.stdin.buffer.readline()
    parts = line.split()

    print("原始数据：", repr(line))
    print("分割结果：", parts)
    print("元素类型：", type(parts[0]).__name__)

原始数据： b'10 20\r\n'
分割结果： [b'10', b'20']
元素类型： bytes


##### 2.2.5：使用 `.splitlines()` 按行分割

- 将一段文本按行分割，返回字符串列表。
- 默认不保留行末换行符。
- 中间的空行会保留为空字符串元素。
- `keepends=True` 可以保留换行符。
- 适合先用 `read()` 读取全文，再按行处理。

In [26]:
text = "10 20\n\nhello\n"

print(text.splitlines())
# ['10 20', '', 'hello']

print(text.splitlines(keepends=True))
# ['10 20\n', '\n', 'hello\n']

['10 20', '', 'hello']
['10 20\n', '\n', 'hello\n']


对比：`.split("\n")` 会因为末尾的分隔符多产生一个 `''`；`.splitlines()` 不会

In [27]:
print(text.split("\n"))
# ['10 20', '', 'hello', '']

['10 20', '', 'hello', '']


In [28]:
with use_input_file(SAMPLE):
    text = sys.stdin.read()
    lines = text.splitlines()

    print(lines)

['10 20', '', 'hello', '中文']


##### 2.2.6：使用 `.strip()` 清理两端空白

- 不传参数时，移除字符串两端的空白，包括空格、制表符和换行符。
- 不会删除字符串中间的空白。
- 返回新的字符串，不修改原字符串。
- 只清理左端用 `.lstrip()`，只清理右端用 `.rstrip()`。

In [29]:
text = "  hello  world \n"

print("原始：", repr(text))
print("两端：", repr(text.strip()))
print("左端：", repr(text.lstrip()))
print("右端：", repr(text.rstrip()))

原始： '  hello  world \n'
两端： 'hello  world'
左端： 'hello  world \n'
右端： '  hello  world'


对比：如果只想去掉行末换行，并保留行末空格

In [30]:
text = "hello  \n"

print(repr(text.strip()))          # 'hello'
print(repr(text.rstrip("\r\n")))    # 'hello  '

'hello'
'hello  '


小结：`.split()` 切分字段，`.splitlines()` 切分行；`.strip()` 只是清理两端字符，不负责分割。 对普通空白分隔输入，直接 `.split()` 通常就够了，无需先 `.strip()`。

### 2.3 转化成需要的数据

##### 2.3.1：转换单个数据的类型

- `int()`：转换成整数，例如 `"42"` → `42`。
- `float()`：转换成浮点数，例如 `"3.14"` → `3.14`。
- `str()`：转换成字符串，例如 `42` → `"42"`。
- 字符串必须符合目标类型的格式，否则会抛出 `ValueError`。
- 转换会返回新值，不会改变原字符串。

In [31]:
text = "42"
number = int(text)

print(repr(text), type(text).__name__)
print(repr(number), type(number).__name__)

price = float("3.14")
print(repr(price), type(price).__name__)

label = str(42)
print(repr(label), type(label).__name__)

'42' str
42 int
3.14 float
'42' str


##### 2.3.2：使用 `map()` 批量转换

- `map(函数, 可迭代对象)` 将函数应用于其中的每个元素。
- `map(int, parts)` 表示逐个执行 `int()` 转换。
- 这里传入的是函数 `int`，不要写成 `int()`。
- Python 3 中，`map()` 返回迭代器，不是列表。
- 转换按需进行，已经取出的元素会被消耗。

In [32]:
parts = ["10", "20", "30"]
numbers = map(int, parts)

print(type(numbers).__name__)  # map

for number in numbers:
    print(number, type(number).__name__)

print(list(numbers))  # []：已经遍历完了

map
10 int
20 int
30 int
[]


##### 2.3.3：将转换结果保存为列表

- `list(map(int, parts))` 将各字段转换成整数并收集到列表中。
- 得到的列表可以通过下标访问，也可以反复遍历。
- 适合题目中的整数数组。
- 读取小数数组时，将 `int` 换成 `float`。

In [33]:
with use_input_file(SAMPLE):
    line = input()
    parts = line.split()
    nums = list(map(int, parts))

    print("读取：", repr(line))
    print("分割：", parts)
    print("转换：", nums)
    print("元素类型：", type(nums[0]).__name__)

读取： '10 20'
分割： ['10', '20']
转换： [10, 20]
元素类型： int


说明：也可以合成一行写

In [34]:
with use_input_file(SAMPLE):
    nums = list(map(int, input().split()))
    print(nums)

[10, 20]


##### 2.3.4：使用解包接收固定数量的数据

- `a, b = ...` 将两个元素分别赋给 `a` 和 `b`。
- 右侧可以直接使用 `map()`，不必先转换成列表。
- 元素数量必须与变量数量一致，否则抛出 `ValueError`。
- 解包本身只负责分配元素，类型转换由 `int()` 等函数完成。

In [35]:
with use_input_file(SAMPLE):
    a, b = map(int, input().split())

    print(a, type(a).__name__)
    print(b, type(b).__name__)
    print("两数之和：", a + b)

10 int
20 int
两数之和： 30


对比：没有类型转换的情况

In [36]:
a, b = "10 20".split()

print(a + b)  # '1020'：字符串拼接

1020


##### 2.3.5：使用列表推导式批量转换

- `[int(x) for x in parts]` 逐个转换元素，并直接生成列表。
- 对于简单整数转换，与 `list(map(int, parts))` 作用相同。
- 当每个元素需要更复杂的处理时，列表推导式通常更方便表达。

In [37]:
parts = "10 20 30".split()

nums1 = list(map(int, parts))
nums2 = [int(x) for x in parts]

print(nums1)
print(nums2)
print(nums1 == nums2)  # True

[10, 20, 30]
[10, 20, 30]
True


小结：最重要的三个模板
| 题目需要的数据 | 写法 |
|---|---|
| 一行一个整数 | `n = int(input())` |
| 一行固定两个整数 | `a, b = map(int, input().split())` |
| 一行若干个整数组成数组 | `nums = list(map(int, input().split()))` |

## 3、常见数据结构的输入

说明：这里不再使用 `input()` 这些命令，而是将需要的数据直接存好，主要展示构建数据结构的步骤。总共有以下数据结构
- 链表
- 完全二叉树
- 普通二叉树
- 一般图

### 3.1 从列表构建单向链表

给定列表 `nums = [1, 2, 3, 4]`，依次创建节点并连接：

head → 1 → 2 → 3 → 4 → None

**节点结构**

- `val`：当前节点保存的值。
- `next`：指向下一个节点的引用，`None` 表示没有后继节点。

**构建过程**

- `head` 保存头节点，`tail` 保存尾节点。
- 创建第一个节点时，让 `head` 和 `tail` 同时指向它。
- 后续通过 `tail.next = node` 接入新节点，再用 `tail = node` 更新尾节点。
- 每次都从尾部添加，因此链表顺序与列表顺序一致。

**遍历过程**

- 使用 `p = head`，从头节点开始访问。
- 通过 `p = p.next` 移动到下一个节点，直到 `p` 为 `None`。
- 移动的是变量 `p`，不会改变 `head`，也不会修改链表。
- 如果 `nums` 为空，`head` 保持为 `None`，得到空链表。

构建和完整遍历的时间复杂度均为 O(n)，新建节点占用 O(n) 空间。

In [38]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


# 固定数据，不涉及输入读取
nums = [1, 2, 3, 4]

# 构建链表
head = None
# tail = None

for x in nums:
    node = ListNode(x)

    if head is None:
        head = tail = node
    else:
        tail.next = node
        tail = node

# 遍历链表
p = head

while p is not None:
    print(p.val)
    p = p.next

1
2
3
4


### 3.2 从列表构建完全二叉树

完全二叉树除最后一层外，其余层都填满；最后一层的节点从左向右连续排列。

列表 `nums` 按层序保存节点值，不包含表示缺失节点的占位符。例如：

```text
nums = [1, 2, 3, 7, 5, 6]

        1
       / \
      2   3
     / \ /
    7  5 6
```

**节点结构**

- `val`：节点值。
- `left`：左孩子，没有时为 `None`。
- `right`：右孩子，没有时为 `None`。

**递归构建**

采用从 0 开始的列表下标：
- 当前节点下标：`idx`。
- 左孩子下标：`2 * idx + 1`。
- 右孩子下标：`2 * idx + 2`。
- 下标越界时返回 `None`，表示该位置没有节点。
- 从下标 0 开始，递归连接左右子树，最终返回根节点。
- 空列表会得到空树，即根节点为 `None`。

**收集叶子节点**

- 叶子节点的左右孩子都为 `None`。
- 递归访问左右子树，收集所有叶子节点的值。
- 示例中收集到 `[7, 5, 6]`，排序后为 `[5, 6, 7]`。
- 排序只改变叶子值列表，不修改树。

完全二叉树描述的是树的形状，对节点值的大小关系没有要求。

In [39]:
class Node:
    def __init__(self, val: int):
        self.val = val
        self.left: Node | None = None
        self.right: Node | None = None


def build_tree(nums, idx):
    if idx >= len(nums):
        return None

    node = Node(nums[idx])
    node.left = build_tree(nums, 2 * idx + 1)
    node.right = build_tree(nums, 2 * idx + 2)

    return node


def collect_leaves(root, leaves):
    if root is None:
        return

    if root.left is None and root.right is None:
        leaves.append(root.val)
        return

    collect_leaves(root.left, leaves)
    collect_leaves(root.right, leaves)


# 固定数据，按层序排列
nums = [1, 2, 3, 7, 5, 6]

# 构建完全二叉树
root = build_tree(nums, 0)

# 收集并排序叶子节点的值
leaves = []
collect_leaves(root, leaves)

print("叶子节点：", leaves)

leaves.sort()
print("排序结果：", leaves)

叶子节点： [7, 5, 6]
排序结果： [5, 6, 7]


### 3.3 从层序列表构建普通二叉树

普通二叉树允许任意位置缺少孩子。本例用 `-1` 表示空节点，因此实际节点值不能是 `-1`。

例如：

```text
nums = [1, -1, 2, 3, 4]

    1
     \
      2
     / \
    3   4
```

**列表的含义**

- 第一个元素表示根节点。
- 后续元素依次为每个非空节点提供左、右孩子。
- `-1` 只表示该孩子为空，不创建节点，也不加入队列。
- 本例中，`1` 的两个孩子是 `-1、2`，接下来直接处理节点 `2` 的孩子 `3、4`。
- 这是紧凑层序表示，不能直接套用完全二叉树的下标公式。

**构建过程**

- 使用队列保存等待分配孩子的非空节点。
- 从队首取出一个节点，依次读取它的左、右孩子。
- 遇到非空孩子时，创建节点、连接父节点，并加入队尾。
- 重复上述过程，直到队列为空或列表读取完毕。
- 末尾未提供的孩子视为 `None`。
- 空列表或根节点标记为 `-1` 时，得到空树。

**层序遍历**

- 将根节点加入队列。
- 每次从队首取出节点并输出，再依次加入非空的左、右孩子。
- 因为队列先进先出，节点会按从上到下、同层从左到右的顺序访问。
- 输出只包含非空节点，不能单凭这个输出还原原来的树形。

In [40]:
from collections import deque


class TreeNode:
    def __init__(self, val: int):
        self.val = val
        self.left: TreeNode | None = None
        self.right: TreeNode | None = None


def build_tree(nums):
    if not nums or nums[0] == -1:
        return None

    root = TreeNode(nums[0])
    queue = deque([root])
    index = 1

    while queue and index < len(nums):
        cur = queue.popleft()

        # 分配左孩子
        if index < len(nums):
            if nums[index] != -1:
                cur.left = TreeNode(nums[index])
                queue.append(cur.left)
            index += 1

        # 分配右孩子
        if index < len(nums):
            if nums[index] != -1:
                cur.right = TreeNode(nums[index])
                queue.append(cur.right)
            index += 1

    return root


# 固定数据，-1 表示空节点
nums = [1, -1, 2, 3, 4]

# 构建普通二叉树
root = build_tree(nums)

# 层序遍历非空节点
bfs = deque()

if root is not None:
    bfs.append(root)

while bfs:
    cur = bfs.popleft()
    print(cur.val)

    if cur.left is not None:
        bfs.append(cur.left)

    if cur.right is not None:
        bfs.append(cur.right)

1
2
3
4


### 3.4 使用邻接矩阵存储图

给定 5 个节点，以及以下无向边：

```text
1—2
1—3
2—4
3—4
```

**存储方式**

- 使用二维列表 `matrix`。
- `matrix[u][v] = 1`：节点 u 和 v 之间有直接边。
- `matrix[u][v] = 0`：没有直接边，不代表两点之间不存在路径。
- 节点编号从 1 开始，因此分配 `(n + 1) × (n + 1)` 的空间，下标 0 不使用。

**无向图建边**

- 同时设置 `matrix[u][v] = 1` 和 `matrix[v][u] = 1`。
- 因此无向图的邻接矩阵关于主对角线对称。
- 本例没有自环，对角线保持为 0。
- 如果是有向边 u → v，只设置 `matrix[u][v]`。

空间复杂度为 O(n²)，判断两点之间是否有直接边的时间复杂度为 O(1)。

In [41]:
# 固定数据
n = 5
edge_list = [(1, 2), (1, 3), (2, 4), (3, 4)]
m = len(edge_list)

# 创建邻接矩阵，下标 0 不使用
matrix = [[0] * (n + 1) for _ in range(n + 1)]

# 无向图：双向建边
for u, v in edge_list:
    matrix[u][v] = 1
    matrix[v][u] = 1

# 输出邻接矩阵
for i in range(1, n + 1):
    print(*matrix[i][1:])

0 1 1 0 0
1 0 0 1 0
1 0 0 1 0
0 1 1 0 0
0 0 0 0 0


### 使用邻接表存储图

**存储方式**

- 使用列表 `graph`，其中每个元素也是一个列表。
- `graph[u]` 保存节点 u 的所有相邻节点。
- 节点编号从 1 开始，因此创建 n + 1 个列表，下标 0 不使用。
- 孤立节点的邻接列表为空列表 `[]`。

**无向图建边**

- 将 v 加入 `graph[u]`，将 u 加入 `graph[v]`。
- 每条无向边会存储两次，分别对应两个方向。
- 如果是有向边 u → v，只执行 `graph[u].append(v)`。
- 邻居的排列顺序取决于添加边的顺序，不会自动排序。

空间复杂度为 O(n + m)，适合遍历某个节点的所有邻居。

In [42]:
# 固定数据，与邻接矩阵示例相同
n = 5
edge_list = [(1, 2), (1, 3), (2, 4), (3, 4)]
m = len(edge_list)

# 创建邻接表，下标 0 不使用
graph = [[] for _ in range(n + 1)]

# 无向图：双向建边
for u, v in edge_list:
    graph[u].append(v)
    graph[v].append(u)

# 输出每个节点的邻居
for u in range(1, n + 1):
    print(f"{u}: {graph[u]}")

1: [2, 3]
2: [1, 4]
3: [1, 4]
4: [2, 3]
5: []


小结：

设 n 为节点数，m 为边数，deg(u) 为节点 u 的邻居数量。

| 比较项 | 邻接矩阵 | 邻接表（列表实现） |
|---|---|---|
| 核心内容 | 每对节点之间是否有边 | 每个节点有哪些邻居 |
| 空间复杂度 | O(n²) | O(n + m) |
| 判断 u、v 是否直接相邻 | O(1) | O(deg(u)) |
| 遍历 u 的所有邻居 | O(n) | O(deg(u)) |
| 常见适用情况 | 图较稠密，频繁查询两点是否相邻 | 图较稀疏，进行 BFS、DFS 等遍历 |

注意：两种结构存储的都是直接相邻关系，不是任意两点之间的路径可达关系。